# Smooth Static to Dynamic Coefficient of Friction

In [ ]:
from sympy import *
import numpy as np
import plotly.graph_objects as go

In [ ]:
import plotly
from IPython.display import display, HTML
# Tomas Mazak's workaround
plotly.offline.init_notebook_mode()
display(HTML(
    '<script type="text/javascript" async src="https://cdnjs.cloudflare.com/ajax/libs/mathjax/2.7.1/MathJax.js?config=TeX-MML-AM_SVG"></script>'
))
##

In [ ]:
x = Symbol('x', real=True)
eps_v = Symbol(r'\epsilon_v', real=True, positive=True)
mu_s, mu_k = symbols(r'\mu_s \mu_k', real=True, positive=True)

In [ ]:
import re


def plot(*fs, title=None, min_x=-2, max_x=2, even=False, odd=False, **subs):
    subs = {eps_v: subs.get(
        "eps_v", 1e-3), mu_s: subs.get("mu_s", 1), mu_k: subs.get("mu_k", 0.1)}
    xs = np.linspace(min_x*subs[eps_v], max_x*subs[eps_v], 201)
    s = np.sign(xs) if odd else np.ones_like(xs)

    def ys(f):
        return s * np.array([f.subs({x: abs(xi) if (even or odd) else xi} | subs) for xi in xs], dtype=float)

    fig = go.Figure(
        [go.Scatter(x=xs, y=ys(f)) for f in fs],
        layout=dict(
            width=800, height=600, template="plotly_dark",
            xaxis_title=r'speed', title=title
        ))
    file_name = re.sub(
        r'[\$\_\{\}\^\\]|\(x\)', '', title).replace("mathrm", "").replace(' ', '_').strip()
    # fig.write_image(f"../docs/source/_static/img/{file_name}.png")
    fig.show()


def print_latex(expr):
    print(latex(expr).replace(r"\\", r"\\" + "\n"))


def print_code(expr):
    print(cxxcode(expr).replace(r"\epsilon", "eps").replace(
        r"\mu", "mu").replace("x", "y"))

## Friction Mollifier

In [ ]:
def f0(x):
    """Smooth friction mollifier"""
    return x * x * (1 - x / (3 * eps_v)) / eps_v + eps_v / 3


sym_f0 = Piecewise(
    (x, x >= eps_v),
    (f0(x), x < eps_v)
)

display(Eq(Symbol("f_{0}(x)"), sym_f0.expand()))
print_latex(sym_f0.expand())

plot(sym_f0, title=r'$f_0(x)$', even=True)

In [ ]:
def f1(x):
    """Derivative of f0"""
    return x * (2 - x / eps_v) / eps_v


# def sign(x):
#     return Piecewise((1, x > 0), (-1, x < 0), (0, True))


sym_f1 = Piecewise(
    (f1(x), x <= eps_v),
    (1, x > eps_v)
)

display(Eq(Symbol("f_{1}(x)"), sym_f1.expand()))

plot(sym_f1, title=r"$f_1(x)$", odd=True)

## Smooth Coefficient of Friction

We can use a polynomial to model a smooth transition from static to kinematic coefficient of friction.

In [ ]:
def mu_cos(x):
    return (mu_k - mu_s) * (0.5 * (-cos(pi * x / eps_v) + 1)) + mu_s


def mu_cubic(x):
    return (mu_k - mu_s) * (3 * (x / eps_v)**2 - 2 * (x / eps_v)**3) + mu_s


def mu_quadratic_part1(x):
    return (2 / eps_v**2) * (mu_k - mu_s) * x**2 + mu_s


def mu_quadratic_part2(x):
    return -2 * (mu_k - mu_s) / eps_v**2 * (x - eps_v)**2 + mu_k


sym_mu = Piecewise(
    (mu_quadratic_part1(x), x <= eps_v / 2),
    (mu_quadratic_part2(x), x <= eps_v),
    (mu_k, True)
).simplify()

display(Eq(Symbol(r"\mu(x)"), sym_mu))

print_latex(sym_mu)

plot(sym_mu, title=r"$\mu(x)$", even=True)
plot(sym_mu.diff(x), title=r"$\mu'(x)$", odd=True)

## Smooth Coefficient of Friction Molliifier
We know we want a smooth force mollifier of $\mu(x)f_1(x)$, so we can integrate this function to get a mollifier of $\int \mu(x)f_1(x)dx$ that produces the desired force.

In [ ]:
def mu_f1_part1(x):
    return mu_quadratic_part1(x) * f1(x)


def mu_f1_part2(x):
    return mu_quadratic_part2(x) * f1(x)


sym_mu_f1 = Piecewise(
    (mu_f1_part1(x), x <= eps_v/2),
    (mu_f1_part2(x), x <= eps_v),
    (mu_k, True)
).simplify()

display(Eq(Symbol(r"\mu(x) f_{1}(x)"), sym_mu_f1))

print_latex(sym_mu_f1)

plot(sym_mu_f1, title=r'$\mu(x) f_1(x)$', odd=True)

The resulting force mollifier is a polynomail in $x$, so we can use sympy to get the exact integral.

In [ ]:
mu_f0_part1 = integrate(mu_f1_part1(x), x)
mu_f0_part2 = integrate(mu_f1_part2(x), x)

# Constant of integration s.t. mu_f0_part2(eps_v) = mu_k * eps_v
c = mu_k * eps_v - mu_f0_part2.subs(x, eps_v)
mu_f0_part2 = mu_f0_part2 + c

# Constant of integration s.t. mu_f0_part1(eps_v/2) = mu_f0_part2(eps_v/2)
c = mu_f0_part2.subs(x, eps_v / 2) - mu_f0_part1.subs(x, eps_v / 2)
mu_f0_part1 = mu_f0_part1 + c

sym_mu_f0 = Piecewise(
    (mu_f0_part1.simplify().subs(x, x), x <= eps_v/2),
    (mu_f0_part2.simplify().subs(x, x), x <= eps_v),
    (mu_k * x, True)
).simplify()

display(Eq(Symbol(r"\int \mu(x) f_1(x) \mathrm{d}x"), sym_mu_f0))

print_latex(sym_mu_f0.simplify())

plot(sym_mu_f0, title=r'$\int \mu(x) f_1(x) \mathrm{d}x$', even=True)

In [ ]:
display(poly(mu_f0_part1, x))
display(poly(mu_f0_part2, x))

In [ ]:
print(cxxcode(horner(poly(mu_f0_part1.subs({x/eps_v: Symbol('z')}), Symbol("z")))).replace(
    r"\epsilon", "eps").replace(r"\mu", "mu"))
print(cxxcode(horner(poly(mu_f0_part2.subs({x/eps_v: Symbol('z')}), Symbol("z")))).replace(
    r"\epsilon", "eps").replace(r"\mu", "mu"))

# Checking my work

In [ ]:
f = mu_f0_part1.simplify()
display(f)
display(f.subs(mu_s, mu_k).simplify().expand())

In [ ]:
f1_over_x = nsimplify((f.diff(x) / x).simplify())
display(f1_over_x)
display(f1_over_x.subs(mu_s, mu_k).simplify().expand())

In [ ]:
smooth_friction_f2_x_minus_f1_over_x3 = nsimplify(
    ((f.diff(x).diff(x) * x - f.diff(x)) / x**3).simplify())
display(smooth_friction_f2_x_minus_f1_over_x3)
display(smooth_friction_f2_x_minus_f1_over_x3.subs(
    mu_s, mu_k).simplify().expand())

# Safe Division

In [ ]:
mu_f2_x_minus_mu_f1_over_x3 = \
    ((mu_f1_part1(x).diff(x) * x - mu_f1_part1(x)) / x**3).simplify()
mu_f2_x_minus_mu_f1_over_x3.simplify()

In [ ]:
mu_f2_x_minus_mu_f1_over_x3.expand()

In [ ]:
print(cxxcode(horner(Poly(mu_f2_x_minus_mu_f1_over_x3.expand().subs(
    {1/eps_v: Symbol('z')}), Symbol("z")))))

In [ ]:
mu_f2_x_minus_mu_f1_over_x3 = \
    ((mu_f1_part2(x).diff(x) * x - mu_f1_part2(x)) / x**3).simplify()
mu_f2_x_minus_mu_f1_over_x3.simplify()

In [ ]:
mu_f2_x_minus_mu_f1_over_x3.expand()

In [ ]:
print(cxxcode(horner(Poly(mu_f2_x_minus_mu_f1_over_x3.expand().subs(
    {1/eps_v: Symbol('z')}), Symbol("z")))))

# Tangential Adhesion

In [ ]:
def a0_part1(x):
    """Smooth adhesion mollifier"""
    return x * x / eps_v * (1 - x / (3 * eps_v))


def a0_part2(x):
    """Smooth adhesion mollifier"""
    return 4 * eps_v / 3


sym_a0 = Piecewise(
    (0, x <= 0),
    (a0_part1(x), abs(x) < 2 * eps_v),
    (a0_part2(x), abs(x) >= 2 * eps_v),
)

display(Eq(Symbol("f_{0}^{a}(x)"), sym_a0.expand()))

plot(sym_a0, title=r'$f_{0}^{a}(x)$', x_offset=-1, max_x=3)

In [ ]:
def a1(x):
    """Derivative of a0"""
    return (x / eps_v) * (2 - x / eps_v)


# def sign(x):
#     return Piecewise((1, x > 0), (-1, x < 0), (0, True))


sym_a1 = Piecewise(
    (0, x < 0),
    (a1(x), x <= 2 * eps_v),
    (0, True),
)

display(Eq(Symbol("f_{1}^{a}(x)"), sym_a1.expand()))

plot(sym_a1, title=r"$f_{1}^{a}(x)$", min_x=-1, max_x=3)

# Smooth Coefficient of Adhesion Mollifier

In [ ]:
def mu_a1_part1(x):
    return mu_quadratic_part1(x) * a1(x)


def mu_a1_part2(x):
    return mu_quadratic_part2(x) * a1(x)


sym_mu_a1 = Piecewise(
    (0, x <= 0),
    (mu_a1_part1(x), x <= eps_v / 2),
    (mu_a1_part2(x), x <= eps_v),
    (mu_k * a1(x), x <= 2 * eps_v),
    (0, True)
).simplify()

# sym_mu_a1 = (sym_mu * sym_a1).simplify()

display(Eq(Symbol(r"\mu(x) f_{1}^{a}(x)"), sym_mu_a1))

plot(sym_mu_a1, title=r'$\mu(x) f_{1}^{a}(x)$', min_x=-1, max_x=3)

In [ ]:
mu_a0_part1 = integrate(mu_a1_part1(x), x)
mu_a0_part2 = integrate(mu_a1_part2(x), x)

# Constant of integration s.t. mu_a0(eps_v) = mu_k * a0_part1(eps_v)
c2 = (mu_k * a0_part1(eps_v) - mu_a0_part2.subs(x, eps_v)).simplify()
c1 = (mu_a0_part2.subs(x, eps_v/2) - mu_a0_part1.subs(x, eps_v/2)).simplify()

mu_a0_part2 -= c1

print(cxxcode(c1.simplify()).replace(
    r"\epsilon_v", "eps_a").replace(r"\mu", "mu"))
print(cxxcode((c2+c1).simplify()).replace(r"\epsilon_v",
      "eps_a").replace(r"\mu", "mu"))

sym_mu_a0 = Piecewise(
    (0, x <= 0),
    (mu_a0_part1, x <= eps_v/2),
    (mu_a0_part2, x <= eps_v),
    ((mu_k * a0_part1(x)).simplify()-c2-c1, x <= 2 * eps_v),
    ((mu_k * a0_part2(x)).simplify()-c2-c1, x > 2 * eps_v),
)

display(Eq(Symbol(r"\int \mu(x) f_{1}^{a}(x) \mathrm{d}x"), sym_mu_a0))

plot(
    sym_mu_a0, title=r'$\int \mu(x) f_{1}^{a}(x) \mathrm{d}x$', min_x=-1, max_x=3)

In [ ]:
display(mu_a0_part1, mu_a0_part2 - c1)

In [ ]:
print_code(horner(poly(mu_a0_part1.subs({x/eps_v: Symbol('z')}), Symbol("z"))))
print_code(horner(poly(mu_a0_part2.subs({x/eps_v: Symbol('z')}), Symbol("z"))))

In [ ]:
a2_part1 = ((mu_a1_part1(x).diff(x) * x - mu_a1_part1(x)) / x**3).simplify()
print_code(horner(Poly(a2_part1.expand().subs(
    {1/eps_v: Symbol('z')}), Symbol('z'))))

In [ ]:
a2_part2 = ((mu_a1_part2(x).diff(x) * x - mu_a1_part2(x)) / x**3).simplify()
print_code(horner(Poly(a2_part2.expand().subs(
    {1/eps_v: Symbol('z')}), Symbol('z'))))

In [ ]:
((sym_mu_a1.diff(x) * x - sym_mu_a1) / x**3).simplify()

In [ ]:
# plot(((sym_mu_a1.diff(x) * x - sym_mu_a1) / x**3).simplify(), title="a2")